In [1]:
from pathlib import Path
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.pyplot as plt
import numpy as np


from data_loader_v2 import align_spikes, get_spike_counts, load_session,load_region_channels, selected_condition_masks


In [2]:
DATA_DIR = Path("../Data")

REGION = "pfc"
N_COLOR_GROUPS = 4

COUNT_WINDOW_SIZE = 0.100
COUNT_STEP = 0.025

PRE_EPOCH = "CUE2_ON"
PRE_ALIGNMENT_WINDOW = (-0.6, -0.2)
PRE_TARGET_CENTER = -0.4


POST_EPOCH = "WHEEL_ON"
POST_ALIGNMENT_WINDOW = (-0.3, 0.1)
POST_TARGET_CENTER = -0.1


idx_center = 6

session_ids = sorted(path.name for path in DATA_DIR.iterdir() if path.is_dir())


## Helpers



In [5]:
def build_condition_average(session_ids, epoch, alignment_window,region):
    rows = []
    for session_id in session_ids:
        print(session_id)
        
        session = load_session(session_id, data_dir=DATA_DIR)

        region_channels = set(load_region_channels(session["session"], region=region,data_dir=DATA_DIR))
        unit_indices = [i for i, unit in enumerate(session["units"]) if unit["channel"] in region_channels]

        if len(session["trials"]) == 0 or len(unit_indices) == 0:
            print("skipped, ", session_id)
            continue

        aligned = align_spikes(session, epoch, window=alignment_window)
        counts = get_spike_counts(aligned, window_size=COUNT_WINDOW_SIZE, step=COUNT_STEP)

        masks = selected_condition_masks(session["trials"],N_COLOR_GROUPS)


        for unit_i in unit_indices:
            unit_counts = counts["counts"][unit_i] / COUNT_WINDOW_SIZE
            condition_values = []
            for mask in masks:
                    condition_values.append(np.nanmean(unit_counts[mask],0))
            rows.append(condition_values)

    return np.array(rows)



In [6]:
cond_means_pre =build_condition_average(session_ids,PRE_EPOCH, PRE_ALIGNMENT_WINDOW, "pfc")
cond_means_post =build_condition_average(session_ids,POST_EPOCH, POST_ALIGNMENT_WINDOW, "pfc")


170709
skipped,  170709
170710
170713
170714
170717
170718
170724
170728
170729
170801
170709
skipped,  170709
170710
170713
170714
170717
170718
170724
170728
170729
170801


In [ ]:
np.save("cond_means_pre",cond_means_pre)
np.save("cond_means_post",cond_means_post)